In [8]:
#!pip install yfinance "hmmlearn>=0.3.2"

In [9]:

import warnings
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import yfinance as yf
from hmmlearn.hmm import CategoricalHMM, GaussianHMM


In [27]:
TICKER = "IBIT"
LOOKBACK = "3mo"
ROLLING_SESSIONS = 21
ANNUALIZATION_DAYS = 252

STATES = ("UP", "FLAT", "DOWN")
STATE_TO_CODE = {state: code for code, state in enumerate(STATES)}
CODE_TO_STATE = {code: state for state, code in STATE_TO_CODE.items()}
K_GRID = np.round(np.arange(0.0, 3.01, 0.25), 2)
MIN_TRAIN_OBSERVATIONS = 20
TUNE_FRACTION = 0.70
RANDOM_SEED = 42
FEATURE_COLUMNS = ("close_return",
# "open_close_return", 
#"intraday_range"
)

In [30]:
# auto_adjust=True applies a consistent split/dividend adjustment to OHLC.
prices = yf.download(
    TICKER,
    period=LOOKBACK,
    interval="1d",
    auto_adjust=True,
    progress=False,
)
if prices.empty:
    raise RuntimeError(f"No daily price data returned for {TICKER}.")

if isinstance(prices.columns, pd.MultiIndex):
    for level in range(prices.columns.nlevels):
        values = prices.columns.get_level_values(level)
        if {"Open", "High", "Low", "Close"}.issubset(set(values)):
            prices.columns = values
            break

prices = prices.loc[:, [ #"Open", "High", "Low", 
                        "Close"]].copy()
prices = prices[~prices.index.duplicated(keep="last")].sort_index().dropna()

# Only use completed sessions; today's candle is not today's signal input.
market_today = pd.Timestamp.now(tz=ZoneInfo("America/New_York")).date()
prices = prices.loc[[session.date() < market_today for session in prices.index]]
if prices.empty:
    raise RuntimeError("No completed daily candles are available in the requested period.")

base = prices.copy()
base["yesterday_return_pct"] = 100*base["Close"].pct_change().shift(1)
#base["open_close_change"] = base["Close"] - base["Open"]
#base["intraday_range"] = (base["High"] - base["Low"]) / base["Open"]

# Shift by one so each candle's threshold uses only returns known beforehand.
rolling_mean_return = base["yesterday_return_pct"].rolling(
    ROLLING_SESSIONS, min_periods=ROLLING_SESSIONS
).mean().shift(1)

rolling_annualized_volatility = (
    base["yesterday_return_pct"]
    .rolling(ROLLING_SESSIONS, min_periods=ROLLING_SESSIONS)
    .std(ddof=1)
    .shift(1)
    * np.sqrt(ANNUALIZATION_DAYS) 
)
base["avg_daily_return_pct"] = rolling_mean_return
base["ann_vol_pct"] = rolling_annualized_volatility

base.sort_values(by="Date", ascending=False).head(45)


Price,Close,yesterday_return_pct,avg_daily_return_pct,ann_vol_pct
Date,,,,
2026-10-16,46.150002,0.756917,0.358654,36.362105
2026-10-15,46.590000,-1.888392,0.275088,38.190696
2026-10-14,46.240002,0.576185,0.353181,38.771713
2026-10-13,47.130001,0.665954,0.331280,38.757695
2026-10-12,46.860001,0.626893,0.235843,39.184561
2026-10-09,46.549999,-2.012287,0.320939,38.373315
2026-10-08,46.259998,-2.639725,0.358203,37.702452
2026-10-07,47.209999,-0.144151,0.250001,38.886978
2026-10-06,48.490002,1.738952,0.445579,43.228748


Price,Open,High,Low,Close,close_return,open_close_change,daily_return_pct,ann_vol_pct
Date,,,,,,,,
2026-10-16,51.849998,52.330002,51.360001,51.509998,-0.007897,-0.340000,-0.246018,16.525035
2026-10-15,51.250000,51.939999,50.970001,51.919998,0.033851,0.669998,-0.414363,16.872134
2026-10-14,51.310001,51.669998,50.009998,50.220001,-0.061133,-1.090000,-0.309062,15.749894
2026-10-13,54.279999,54.340000,53.180000,53.490002,-0.015823,-0.789997,-0.239756,15.212952
2026-10-12,54.720001,54.740002,54.299999,54.349998,-0.007850,-0.370003,-0.257017,15.394708
2026-10-09,54.900002,55.250000,54.750000,54.779999,0.024883,-0.120003,-0.321599,16.054762
2026-10-08,53.340000,53.639999,52.840000,53.450001,-0.006875,0.110001,-0.366450,15.996027
2026-10-07,53.900002,54.389999,53.599998,53.820000,-0.029396,-0.080002,-0.399151,16.054871
2026-10-06,55.299999,55.860001,54.860001,55.450001,0.005804,0.150002,-0.358943,16.750499


In [ ]:
def make_labeled_data(k: float) -> pd.DataFrame:
    """Create threshold states using only prior-session rolling statistics."""
    data = base.copy()
    data["threshold"] = (
        abs(k)
        * data["rolling_mean_return"].abs()
        / data["rolling_annualized_volatility"]
        * data["Open"]
    )
    usable = data.dropna(
        subset=["threshold", "rolling_mean_return", "rolling_annualized_volatility", *FEATURE_COLUMNS]
    ).copy()
    usable = usable.loc[usable["rolling_annualized_volatility"] > 0]
    change = usable["open_close_change"]
    usable["state"] = np.select(
        [change > usable["threshold"], change < -usable["threshold"]],
        ["UP", "DOWN"],
        default="FLAT",
    )
    usable["state_code"] = usable["state"].map(STATE_TO_CODE).astype(int)
    return usable


def balanced_accuracy(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Mean recall over classes present in the actual labels."""
    recalls = []
    for code in np.unique(actual):
        mask = actual == code
        recalls.append(float(np.mean(predicted[mask] == code)))
    return float(np.mean(recalls)) if recalls else float("nan")


def categorical_hmm_forecast(train_codes: np.ndarray):
    """Fit on threshold states and forecast the next mapped latent state."""
    observations = np.asarray(train_codes, dtype=int).reshape(-1, 1)
    model = CategoricalHMM(
        n_components=3,
        n_features=len(STATES),
        n_iter=150,
        tol=1e-3,
        random_state=RANDOM_SEED,
    )
    model.fit(observations)
    filtered_hidden = model.predict_proba(observations)[-1]
    next_hidden = filtered_hidden @ model.transmat_

    # Rank hidden states by expected directional observation: UP, FLAT, DOWN.
    expected_direction = model.emissionprob_ @ np.array([1.0, 0.0, -1.0])
    latent_order = np.argsort(expected_direction)[::-1]
    state_probabilities = next_hidden[latent_order]
    state_probabilities = state_probabilities / state_probabilities.sum()
    transition_matrix = model.transmat_[np.ix_(latent_order, latent_order)]
    return model, state_probabilities, transition_matrix


def gaussian_hmm_forecast(train_features: np.ndarray):
    """Fit OHLC features and map latent states by mean open-to-close return."""
    values = np.asarray(train_features, dtype=float)
    center = values.mean(axis=0)
    scale = values.std(axis=0, ddof=0)
    scale[scale < 1e-12] = 1.0
    standardized = (values - center) / scale

    model = GaussianHMM(
        n_components=3,
        covariance_type="diag",
        n_iter=150,
        tol=1e-3,
        min_covar=1e-4,
        random_state=RANDOM_SEED,
    )
    model.fit(standardized)
    filtered_hidden = model.predict_proba(standardized)[-1]
    next_hidden = filtered_hidden @ model.transmat_

    direction_column = FEATURE_COLUMNS.index("open_close_return")
    latent_order = np.argsort(model.means_[:, direction_column])[::-1]
    state_probabilities = next_hidden[latent_order]
    state_probabilities = state_probabilities / state_probabilities.sum()
    transition_matrix = model.transmat_[np.ix_(latent_order, latent_order)]
    return model, state_probabilities, transition_matrix


def probability_row(probabilities: np.ndarray) -> pd.Series:
    return pd.Series(probabilities, index=STATES, name="Next-session probability")


# Tune k using only expanding-window predictions in the first 70% of observations.
example_data = make_labeled_data(float(K_GRID[0]))
observation_count = len(example_data)
tune_end = int(np.floor(observation_count * TUNE_FRACTION))
if tune_end <= MIN_TRAIN_OBSERVATIONS or observation_count - tune_end < 3:
    raise ValueError(
        f"Only {observation_count} labeled sessions remain after the rolling window; "
        "there is not enough data for the requested walk-forward tune/holdout split."
    )
if observation_count < 50:
    warnings.warn(
        f"Only {observation_count} labeled sessions remain. Three months of prices "
        "are a very small sample for three-state HMM estimation.",
        RuntimeWarning,
    )

k_results = []
for k in K_GRID:
    data = make_labeled_data(float(k))
    codes = data["state_code"].to_numpy()
    validation_actual = []
    validation_predicted = []
    failures = 0
    for next_index in range(MIN_TRAIN_OBSERVATIONS, tune_end):
        try:
            _, probabilities, _ = categorical_hmm_forecast(codes[:next_index])
            validation_actual.append(codes[next_index])
            validation_predicted.append(int(np.argmax(probabilities)))
        except (ValueError, FloatingPointError, np.linalg.LinAlgError):
            failures += 1
    if validation_actual:
        k_results.append(
            {
                "k": float(k),
                "balanced_accuracy": balanced_accuracy(
                    np.asarray(validation_actual), np.asarray(validation_predicted)
                ),
                "validation_predictions": len(validation_actual),
                "failed_folds": failures,
            }
        )

if not k_results:
    raise RuntimeError("No k value produced a valid categorical-HMM tuning result.")
k_results = sorted(k_results, key=lambda result: (-result["balanced_accuracy"], result["k"]))
k_table = pd.DataFrame(k_results)
best_k = float(k_table.iloc[0]["k"])
print(f"Data source: Yahoo Finance ({TICKER}, adjusted daily OHLC)")
print(f"Date range: {prices.index.min().date()} through {prices.index.max().date()}")
print(f"Labeled sessions: {observation_count}; tuning ends before row {tune_end}")
print(f"Selected k from tuning data: {best_k:.2f}")
display(k_table)


# Evaluate both models once on the untouched final 30% using expanding training prefixes.
selected_data = make_labeled_data(best_k)
state_codes = selected_data["state_code"].to_numpy()
feature_values = selected_data.loc[:, FEATURE_COLUMNS].to_numpy(dtype=float)
test_actual = []
test_predictions = {"Categorical HMM": [], "Gaussian HMM": [], "Persistence": []}
for next_index in range(tune_end, observation_count):
    _, categorical_probabilities, _ = categorical_hmm_forecast(state_codes[:next_index])
    _, gaussian_probabilities, _ = gaussian_hmm_forecast(feature_values[:next_index])
    test_actual.append(state_codes[next_index])
    test_predictions["Categorical HMM"].append(int(np.argmax(categorical_probabilities)))
    test_predictions["Gaussian HMM"].append(int(np.argmax(gaussian_probabilities)))
    test_predictions["Persistence"].append(int(state_codes[next_index - 1]))

test_actual = np.asarray(test_actual)
metric_rows = []
for model_name, predictions in test_predictions.items():
    predicted = np.asarray(predictions)
    metric_rows.append(
        {
            "model": model_name,
            "balanced_accuracy": balanced_accuracy(test_actual, predicted),
            "test_sessions": len(test_actual),
        }
    )
    confusion = pd.DataFrame(0, index=STATES, columns=STATES)
    for actual_code, predicted_code in zip(test_actual, predicted):
        confusion.loc[CODE_TO_STATE[int(actual_code)], CODE_TO_STATE[int(predicted_code)]] += 1
    print(f"\n{model_name} holdout confusion matrix (rows=actual, columns=predicted):")
    display(confusion)
print("Holdout balanced accuracy:")
display(pd.DataFrame(metric_rows).set_index("model"))


# Refit on all available labeled rows for the latest completed-session state and forecast.
latest_model_a, latest_a_probabilities, latest_a_transition = categorical_hmm_forecast(state_codes)
latest_model_b, latest_b_probabilities, latest_b_transition = gaussian_hmm_forecast(feature_values)
print(f"Latest completed-session state: {selected_data['state'].iloc[-1]}")
print("Next-session probabilities, categorical HMM:")
display(probability_row(latest_a_probabilities))
print("Next-session probabilities, Gaussian HMM:")
display(probability_row(latest_b_probabilities))
print("Categorical HMM transition probabilities (latent states mapped to UP/FLAT/DOWN):")
display(pd.DataFrame(latest_a_transition, index=STATES, columns=STATES))
print("Gaussian HMM transition probabilities (latent states mapped to UP/FLAT/DOWN):")
display(pd.DataFrame(latest_b_transition, index=STATES, columns=STATES))